In [ ]:
import numpy as np
import psi4

# Memory specification
psi4.set_memory(int(5e8))
numpy_memory = 2

mol = psi4.geometry("""
    C    0.000000    0.000000    0.000000
    O    0.000000    0.000000    1.203000
    H    0.000000    0.934000   -0.582000
    H    0.000000   -0.934000   -0.582000
    """)

# Set computation options
psi4.set_options({'basis': 'sto-3g', 'puream': 0})

wfn = psi4.core.Wavefunction.build(mol, psi4.core.get_global_option('basis'))
mints = psi4.core.MintsHelper(wfn.basisset())
basis = wfn.basisset()
n_ao = basis.nao()

In [2]:
def get_overlap_integrals(PA, PB, zeta, AMa, AMb):
    # The first dimension (size 3) automatically vectorizes across [x, y, z] axes
    overlap_integrals = np.zeros((3, AMa + 1, AMb + 1))
    
    overlap_integrals[:, 0, 0] = 1.0

    for ib in range(AMb + 1):
        for ia in range(AMa + 1):
            if ia == 0 and ib == 0:
                continue
                
            # If ib is 0, we are stepping up Center A
            if ib == 0:
                term1 = PA * overlap_integrals[:, ia - 1, 0]
                term2 = (0.5 / zeta) * (ia - 1) * overlap_integrals[:, ia - 2, 0] if ia > 1 else 0.0
                
                overlap_integrals[:, ia, 0] = term1 + term2

            # If ib > 0, we are stepping up Center B
            else:
                term1 = PB * overlap_integrals[:, ia, ib - 1]
                term2 = (0.5 / zeta) * ia * overlap_integrals[:, ia - 1, ib - 1] if ia > 0 else 0.0
                term3 = (0.5 / zeta) * (ib - 1) * overlap_integrals[:, ia, ib - 2] if ib > 1 else 0.0
                
                overlap_integrals[:, ia, ib] = term1 + term2 + term3

    return overlap_integrals

In [3]:
def get_kinetic_integrals(PA, PB, expa, expb, AMa, AMb, SI):
    zeta = expa + expb
    xi = (expa * expb) / zeta
    # The first dimension (size 3) automatically vectorizes across [x, y, z] axes
    kinetic_integrals = np.zeros((3, AMa + 1, AMb + 1))

    # Seed base case
    kinetic_integrals[:, 0, 0] = xi * (1.0 - 2.0 * xi * (PA - PB)**2)

    for ib in range(AMb + 1):
        for ia in range(AMa + 1):
            if ia == 0 and ib == 0:
                continue

            # If ib is 0, stepping up Center A
            if ib == 0:
                term1 = PA * kinetic_integrals[:, ia - 1, ib]
                term2 = (0.5 / zeta) * (ia - 1) * kinetic_integrals[:, ia - 2, ib] if ia > 1 else 0.0
                term3 = (0.5 / zeta) * ib * kinetic_integrals[:, ia - 1, ib - 1] if ib > 0 else 0.0
                term4 = (2 * xi) * (SI[:, ia, ib] - (0.5 / expa) * (ia - 1) * SI[:, ia - 2, ib])

                kinetic_integrals[:, ia, 0] = term1 + term2 + term3 + term4

            # If ib > 0, stepping up Center B
            else:
                term1 = PB * kinetic_integrals[:, ia, ib - 1]
                term2 = (0.5 / zeta) * ia * kinetic_integrals[:, ia - 1, ib - 1] if ia > 0 else 0.0
                term3 = (0.5 / zeta) * (ib - 1) * kinetic_integrals[:, ia, ib - 2] if ib > 1 else 0.0
                term4 = (2 * xi) * (SI[:, ia, ib] - (0.5 / expb) * (ib - 1) * SI[:, ia, ib - 2])

                kinetic_integrals[:, ia, ib] = term1 + term2 + term3 + term4

    return kinetic_integrals

In [4]:
def boys_function(m, x):
    """Evaluates the Boys function F_m(x) using Scipy's error function."""
    if x < 1e-9:
        return 1.0 / (2 * m + 1)
   
    from scipy.special import gammainc, gamma
    return 0.5 * x**(-(m + 0.5)) * gamma(m + 0.5) * gammainc(m + 0.5, x)


def get_NAIs(PA, PB, PC, zeta, AMa, AMb):
    max_m = AMa + AMb
    # Shape: (AMa+1, AMa+1, AMa+1, AMb+1, AMb+1, AMb+1, max_m+1)
    NAI = np.zeros((AMa + 1, AMa + 1, AMa + 1, AMb + 1, AMb + 1, AMb + 1, max_m + 1))
    
    def swap_NAI(swap_dict: dict):
        axis_dict = {
            "ax": 0, "ay": 1, "az": 2,
            "bx": 3, "by": 4, "bz": 5, 
            "m": 6
        }
        
        # Captures current loop variables from the enclosing scope
        curr_idx = [ax, ay, az, bx, by, bz, m]
        for axis, swap in swap_dict.items():
            idx = axis_dict[axis]
            curr_idx[idx] += swap

        if any(idx < 0 for idx in curr_idx):
            return 0.0

        return NAI[tuple(curr_idx)]
    
    # 1. Foundation: Seed the [0,0,0|0,0,0]^(m) base cases
    U = zeta * np.sum(PC**2)
    for m_idx in range(max_m + 1):
        NAI[0, 0, 0, 0, 0, 0, m_idx] = boys_function(m_idx, U) 
        
    # 2. Chained 3D Step-Up Engine
    # We must build L strictly from 1 up to (AMa + AMb) to ensure lower dependencies exist
    for L in range(1, AMa + AMb + 1):
        for ax in range(AMa + 1):
            for ay in range(AMa + 1 - ax):
                for az in range(AMa + 1 - ax - ay):
                    for bx in range(AMb + 1):
                        for by in range(AMb + 1 - bx):
                            for bz in range(AMb + 1 - bx - by):
                                
                                # Only process states that sum to the current total angular momentum
                                if ax + ay + az + bx + by + bz != L:
                                    continue
                                    
                                # The +1 ensures we reach m=0 for the final target integral
                                for m in range(max_m - L + 1):
                                    
                                    # Step up the first available non-zero angular momentum component
                                    if ax > 0:
                                        term1 = PA[0] * swap_NAI({"ax": -1})
                                        term2 = PC[0] * swap_NAI({"ax": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (ax - 1) * (swap_NAI({"ax": -2}) - swap_NAI({"ax": -2, "m": +1})) if ax > 1 else 0.0
                                        term4 = (0.5 / zeta) * bx * (swap_NAI({"ax": -1, "bx": -1}) - swap_NAI({"ax": -1, "bx": -1, "m": +1})) if bx > 0 else 0.0
                                        NAI[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3 + term4
                                        
                                    elif ay > 0:
                                        term1 = PA[1] * swap_NAI({"ay": -1})
                                        term2 = PC[1] * swap_NAI({"ay": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (ay - 1) * (swap_NAI({"ay": -2}) - swap_NAI({"ay": -2, "m": +1})) if ay > 1 else 0.0
                                        term4 = (0.5 / zeta) * by * (swap_NAI({"ay": -1, "by": -1}) - swap_NAI({"ay": -1, "by": -1, "m": +1})) if by > 0 else 0.0
                                        NAI[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3 + term4
                                        
                                    elif az > 0:
                                        term1 = PA[2] * swap_NAI({"az": -1})
                                        term2 = PC[2] * swap_NAI({"az": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (az - 1) * (swap_NAI({"az": -2}) - swap_NAI({"az": -2, "m": +1})) if az > 1 else 0.0
                                        term4 = (0.5 / zeta) * bz * (swap_NAI({"az": -1, "bz": -1}) - swap_NAI({"az": -1, "bz": -1, "m": +1})) if bz > 0 else 0.0
                                        NAI[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3 + term4
                                        
                                    elif bx > 0:
                                        term1 = PB[0] * swap_NAI({"bx": -1})
                                        term2 = PC[0] * swap_NAI({"bx": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (bx - 1) * (swap_NAI({"bx": -2}) - swap_NAI({"bx": -2, "m": +1})) if bx > 1 else 0.0
                                        NAI[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3
                                        
                                    elif by > 0:
                                        term1 = PB[1] * swap_NAI({"by": -1})
                                        term2 = PC[1] * swap_NAI({"by": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (by - 1) * (swap_NAI({"by": -2}) - swap_NAI({"by": -2, "m": +1})) if by > 1 else 0.0
                                        NAI[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3
                                        
                                    elif bz > 0:
                                        term1 = PB[2] * swap_NAI({"bz": -1})
                                        term2 = PC[2] * swap_NAI({"bz": -1, "m": +1})
                                        term3 = (0.5 / zeta) * (bz - 1) * (swap_NAI({"bz": -2}) - swap_NAI({"bz": -2, "m": +1})) if bz > 1 else 0.0
                                        NAI[ax, ay, az, bx, by, bz, m] = term1 - term2 + term3

    return NAI

In [5]:
def angular_combinations(n: int):
    coms = []
    # Psi4 orders Cartesian components from highest z-power down to x
    for nx in range(n, -1, -1):
        for ny in range(n - nx, -1, -1):
            nz = n - nx - ny
            coms.append((nx, ny, nz))
    return coms

In [6]:
S = np.zeros((n_ao,n_ao))
T = np.zeros((n_ao,n_ao))
V = np.zeros((n_ao,n_ao))

# loop over the shells, basis.nshell is the number of shells
for a in range(basis.nshell()):
    for b in range(basis.nshell()):
        # basis.shell is a shell (1s, 2s, 2p, etc.)
        # for water, there are 5 shells: (H: 1s, H: 1s, O: 1s, 2s, 2p)
        ashell = basis.shell(a) 
        bshell = basis.shell(b)

        AMa = ashell.am  # angular momenta
        AMb = bshell.am

        # Basis function index where the shell begins
        a_idx = ashell.function_index  
        b_idx = bshell.function_index

        a_coms = angular_combinations(AMa)
        b_coms = angular_combinations(AMb)

        counta = len(a_coms)
        countb = len(b_coms)
        
        if b < a:
            S[a_idx:a_idx+counta, b_idx:b_idx+countb] = S[b_idx:b_idx+countb, a_idx:a_idx+counta].transpose((1, 0))
            T[a_idx:a_idx+counta, b_idx:b_idx+countb] = T[b_idx:b_idx+countb, a_idx:a_idx+counta].transpose((1, 0))
            V[a_idx:a_idx+counta, b_idx:b_idx+countb] = V[b_idx:b_idx+countb, a_idx:a_idx+counta].transpose((1, 0))
            continue

        # defining centers for each basis function 
        # mol.x() returns the x coordinate of the atom given by ashell.ncenter
        # we use this to define a coordinate vector for our centers
        A = np.array([mol.x(ashell.ncenter), mol.y(ashell.ncenter), mol.z(ashell.ncenter)])
        B = np.array([mol.x(bshell.ncenter), mol.y(bshell.ncenter), mol.z(bshell.ncenter)])

        # each shell has some number of primitives which make up each component of a shell
        # sto-3g has 3 primitives for every component of every shell.
        nprima = ashell.nprimitive 
        nprimb = bshell.nprimitive

        # loop over the primitives within a shell
        for a_prim in range(nprima):  
            for b_prim in range(nprimb):
                expa = ashell.exp(a_prim) 
                expb = bshell.exp(b_prim)

                coefa = ashell.coef(a_prim)
                coefb = bshell.coef(b_prim)
                
                zeta = expa + expb
                xi = (expa * expb) / zeta
                P = (expa * A + expb * B) / zeta
                PA = P - A
                PB = P - B
                AB = A - B

                S_base = (np.pi / zeta)**(3 / 2) * np.exp(-xi * (AB[0]**2 + AB[1]**2 + AB[2]**2))

                # Missing boys function factor, multipled inside get_NA_integrals()
                V_base = 2 * (zeta / np.pi)**0.5 * S_base

                # Calculates an additional layer for kinetic terms
                SI = get_overlap_integrals(PA, PB, zeta, AMa, AMb)
                TI = get_kinetic_integrals(PA, PB, expa, expb, AMa, AMb, SI)

                for counta, a_com in enumerate(a_coms):
                    for countb, b_com in enumerate(b_coms):
                        x, y, z = 0, 1, 2
                        ax, ay, az = a_com
                        bx, by, bz = b_com
                        
                        # 3D Overlap is purely multiplicative: Sx * Sy * Sz
                        S[a_idx + counta, b_idx + countb] += S_base \
                                                           * coefa * coefb \
                                                           * SI[x, ax, bx] \
                                                           * SI[y, ay, by] \
                                                           * SI[z, az, bz]

                        # 3D Kinetic Energy requires the Laplacian cross-terms:
                        # Tx*Sy*Sz + Sx*Ty*Sz + Sx*Sy*Tz
                        t_term = (TI[x, ax, bx] * SI[y, ay, by] * SI[z, az, bz]) + \
                                 (SI[x, ax, bx] * TI[y, ay, by] * SI[z, az, bz]) + \
                                 (SI[x, ax, bx] * SI[y, ay, by] * TI[z, az, bz])

                        T[a_idx + counta, b_idx + countb] += S_base \
                                                           * coefa * coefb \
                                                           * t_term
                        
                        for C_idx in range(mol.natom()):
                            Z_C = mol.Z(C_idx)
                            C = np.array([mol.x(C_idx), mol.y(C_idx), mol.z(C_idx)])
                            PC = P - C
                            
                            VI = get_NAIs(PA, PB, PC, zeta, AMa, AMb)

                            # Multiply by -Z_C, do not cube VI, and strictly query m=0
                            V[a_idx + counta, b_idx + countb] += (-Z_C) * V_base \
                                                                * coefa \
                                                                * coefb \
                                                                * VI[ax, ay, az, bx, by, bz, 0]
                            

                        """# Analytical unrolling approach to compute kinetic integrals (psi4numpy approach)
                        # More expensive than OS scheme as it requires an additional layer of AM to be computed:
                        
                        SI = get_overlap_integrals(PA, PB, zeta, AMa+1, AMb+1)
                        Tx = (1 / 2) * (ax * bx * SI[x, ax - 1, bx - 1] + 4 * expa * expb * SI[x, ax + 1, bx + 1] \
                                - 2 * expa * bx * SI[x, ax + 1, bx - 1] - 2 * expb * ax * SI[x, ax - 1, bx + 1])   \
                                * SI[y, ay, by] * SI[z, az, bz]

                        Ty = (1 / 2) * (ay * by * SI[y, ay - 1, by - 1] + 4 * expa * expb * SI[y, ay + 1, by + 1] \
                                - 2 * expa * by * SI[y, ay + 1, by - 1] - 2 * expb * ay * SI[y, ay - 1, by + 1])   \
                                * SI[x, ax, bx] * SI[z, az, bz]

                        Tz = (1 / 2) * (az * bz * SI[z, az - 1, bz - 1] + 4 * expa * expb * SI[z, az + 1, bz + 1] \
                                - 2 * expa * bz * SI[z, az + 1, bz - 1] - 2 * expb * az * SI[z, az - 1, bz + 1])   \
                                * SI[x, ax, bx] * SI[y, ay, by]

                        T[i_idx + counta, j_idx + countb] += start * coefa * coefb * (Tx + Ty + Tz)"""

In [7]:
def get_ERIs(A, B, C, D, expa, expb, expc, expd, AMa, AMb, AMc, AMd):
    max_m = AMa + AMb + AMc + AMd
    
    ERI = np.zeros((AMa + 1, AMa + 1, AMa + 1, 
                  AMb + 1, AMb + 1, AMb + 1,
                  AMc + 1, AMc + 1, AMc + 1,
                  AMd + 1, AMd + 1, AMd + 1,
                  max_m + 1))

    def swap_ERI(swap_dict: dict):
        axis_dict = {
            "ax": 0, "ay": 1, "az": 2,
            "bx": 3, "by": 4, "bz": 5,
            "cx": 6, "cy": 7, "cz": 8,
            "dx": 9, "dy": 10, "dz": 11, 
            "m": 12
        }
        
        # Captures current loop variables from the enclosing scope
        curr_idx = [ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m]
        for axis, swap in swap_dict.items():
            idx = axis_dict[axis]
            curr_idx[idx] += swap

        if any(idx < 0 for idx in curr_idx):
            return 0.0

        return ERI[tuple(curr_idx)]


    zeta = expa + expb
    eta = expc + expd
    ez_sum = zeta + eta

    P = (expa * A + expb * B) / zeta
    Q = (expc * C + expd * D) / eta

    W = (zeta * P + eta * Q) / ez_sum
    WP = W - P
    WQ = W - Q

    PA = P - A
    PB = P - B
    QC = Q - C
    QD = Q - D
    
    # 1. Foundation: Seed base cases
    rho = (zeta * eta) / (zeta + eta)
    T = rho * np.sum((P-Q)**2)
    for m in range(max_m + 1):
        ERI[0, 0, 0, 
          0, 0, 0,
          0, 0, 0, 
          0, 0, 0,  
          m] = boys_function(m, T)
        
    # 2. Chained 3D Step-Up Engine
    # We must build L strictly from 1 up to (AMa + AMb) to ensure lower dependencies exist
    for L in range(1, AMa + AMb + AMc + AMd + 1):
        for ax in range(AMa + 1):
            for ay in range(AMa + 1 - ax):
                for az in range(AMa + 1 - ax - ay):
                    for bx in range(AMb + 1):
                        for by in range(AMb + 1 - bx):
                            for bz in range(AMb + 1 - bx - by):
                                for cx in range(AMc + 1):
                                    for cy in range(AMc + 1 - cx):
                                        for cz in range(AMc + 1 - cx - cy):
                                            for dx in range(AMd + 1):
                                                for dy in range(AMd + 1 - dx):
                                                    for dz in range(AMd + 1 - dx - dy):
                                
                                                        # Only process states that sum to the current total angular momentum
                                                        if ax + ay + az + bx + by + bz + cx + cy + cz + dx + dy + dz != L:
                                                            continue
                                                            
                                                        # The +1 ensures we reach m=0 for the final target integral
                                                        for m in range(max_m - L + 1):
                                                        # Step up the first available non-zero angular momentum component
                                                            if ax > 0:
                                                                term1 = PA[0] * swap_ERI({"ax": -1})
                                                                term2 = WP[0] * swap_ERI({"ax": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * (ax - 1) * (swap_ERI({"ax": -2}) - (eta / ez_sum) * swap_ERI({"ax": -2, "m": +1})) if ax > 1 else 0.0
                                                                term4 = (0.5 / zeta) * bx * (swap_ERI({"ax": -1, "bx": -1}) - (eta / ez_sum) * swap_ERI({"ax": -1, "bx": -1, "m": +1})) if bx > 0 else 0.0
                                                                term5 = (0.5 / ez_sum) * cx * swap_ERI({"ax": -1, "cx": -1, "m": +1}) if cx > 0 else 0.0
                                                                term6 = (0.5 / ez_sum) * dx * swap_ERI({"ax": -1, "dx": -1, "m": +1}) if dx > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif ay > 0:
                                                                term1 = PA[1] * swap_ERI({"ay": -1})
                                                                term2 = WP[1] * swap_ERI({"ay": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * (ay - 1) * (swap_ERI({"ay": -2}) - (eta / ez_sum) * swap_ERI({"ay": -2, "m": +1})) if ay > 1 else 0.0
                                                                term4 = (0.5 / zeta) * by * (swap_ERI({"ay": -1, "by": -1}) - (eta / ez_sum) * swap_ERI({"ay": -1, "by": -1, "m": +1})) if by > 0 else 0.0
                                                                term5 = (0.5 / ez_sum) * cy * swap_ERI({"ay": -1, "cy": -1, "m": +1}) if cy > 0 else 0.0
                                                                term6 = (0.5 / ez_sum) * dy * swap_ERI({"ay": -1, "dy": -1, "m": +1}) if dy > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif az > 0:
                                                                term1 = PA[2] * swap_ERI({"az": -1})
                                                                term2 = WP[2] * swap_ERI({"az": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * (az - 1) * (swap_ERI({"az": -2}) - (eta / ez_sum) * swap_ERI({"az": -2, "m": +1})) if az > 1 else 0.0
                                                                term4 = (0.5 / zeta) * bz * (swap_ERI({"az": -1, "bz": -1}) - (eta / ez_sum) * swap_ERI({"az": -1, "bz": -1, "m": +1})) if bz > 0 else 0.0
                                                                term5 = (0.5 / ez_sum) * cz * swap_ERI({"az": -1, "cz": -1, "m": +1}) if cz > 0 else 0.0
                                                                term6 = (0.5 / ez_sum) * dz * swap_ERI({"az": -1, "dz": -1, "m": +1}) if dz > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif bx > 0:
                                                                term1 = PB[0] * swap_ERI({"bx": -1})
                                                                term2 = WP[0] * swap_ERI({"bx": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * ax * (swap_ERI({"ax": -1, "bx": -1}) - (eta / ez_sum) * swap_ERI({"ax": -1, "bx": -1, "m": +1})) if ax > 0 else 0.0
                                                                term4 = (0.5 / zeta) * (bx - 1) * (swap_ERI({"bx": -2}) - (eta / ez_sum) * swap_ERI({"bx": -2, "m": +1})) if bx > 1 else 0.0
                                                                term5 = (0.5 / ez_sum) * cx * swap_ERI({"bx": -1, "cx": -1, "m": +1}) if cx > 0 else 0.0
                                                                term6 = (0.5 / ez_sum) * dx * swap_ERI({"bx": -1, "dx": -1, "m": +1}) if dx > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif by > 0:
                                                                term1 = PB[1] * swap_ERI({"by": -1})
                                                                term2 = WP[1] * swap_ERI({"by": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * ay * (swap_ERI({"ay": -1, "by": -1}) - (eta / ez_sum) * swap_ERI({"ay": -1, "by": -1, "m": +1})) if ay > 0 else 0.0
                                                                term4 = (0.5 / zeta) * (by - 1) * (swap_ERI({"by": -2}) - (eta / ez_sum) * swap_ERI({"by": -2, "m": +1})) if by > 1 else 0.0
                                                                term5 = (0.5 / ez_sum) * cy * swap_ERI({"by": -1, "cy": -1, "m": +1}) if cy > 0 else 0.0
                                                                term6 = (0.5 / ez_sum) * dy * swap_ERI({"by": -1, "dy": -1, "m": +1}) if dy > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif bz > 0:
                                                                term1 = PB[2] * swap_ERI({"bz": -1})
                                                                term2 = WP[2] * swap_ERI({"bz": -1, "m": +1})
                                                                term3 = (0.5 / zeta) * az * (swap_ERI({"az": -1, "bz": -1}) - (eta / ez_sum) * swap_ERI({"az": -1, "bz": -1, "m": +1})) if az > 0 else 0.0
                                                                term4 = (0.5 / zeta) * (bz - 1) * (swap_ERI({"bz": -2}) - (eta / ez_sum) * swap_ERI({"bz": -2, "m": +1})) if bz > 1 else 0.0
                                                                term5 = (0.5 / ez_sum) * cz * swap_ERI({"bz": -1, "cz": -1, "m": +1}) if cz > 0 else 0.0
                                                                term6 = (0.5 / ez_sum) * dz * swap_ERI({"bz": -1, "dz": -1, "m": +1}) if dz > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif cx > 0:
                                                                term1 = QC[0] * swap_ERI({"cx": -1})
                                                                term2 = WQ[0] * swap_ERI({"cx": -1, "m": +1})
                                                                term3 = (0.5 / ez_sum) * ax * swap_ERI({"ax": -1, "cx": -1, "m": +1}) if ax > 0 else 0.0
                                                                term4 = (0.5 / ez_sum) * bx * swap_ERI({"bx": -1, "cx": -1, "m": +1}) if bx > 0 else 0.0
                                                                term5 = (0.5 / eta) * (cx - 1) * (swap_ERI({"cx": -2}) - (zeta / ez_sum) * swap_ERI({"cx": -2, "m": +1})) if cx > 1 else 0.0
                                                                term6 = (0.5 / eta) * dx * (swap_ERI({"cx": -1, "dx": -1}) - (zeta / ez_sum) * swap_ERI({"cx": -1, "dx": -1, "m": +1})) if dx > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif cy > 0:
                                                                term1 = QC[1] * swap_ERI({"cy": -1})
                                                                term2 = WQ[1] * swap_ERI({"cy": -1, "m": +1})
                                                                term3 = (0.5 / ez_sum) * ay * swap_ERI({"ay": -1, "cy": -1, "m": +1}) if ay > 0 else 0.0
                                                                term4 = (0.5 / ez_sum) * by * swap_ERI({"by": -1, "cy": -1, "m": +1}) if by > 0 else 0.0
                                                                term5 = (0.5 / eta) * (cy - 1) * (swap_ERI({"cy": -2}) - (zeta / ez_sum) * swap_ERI({"cy": -2, "m": +1})) if cy > 1 else 0.0
                                                                term6 = (0.5 / eta) * dy * (swap_ERI({"cy": -1, "dy": -1}) - (zeta / ez_sum) * swap_ERI({"cy": -1, "dy": -1, "m": +1})) if dy > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif cz > 0:
                                                                term1 = QC[2] * swap_ERI({"cz": -1})
                                                                term2 = WQ[2] * swap_ERI({"cz": -1, "m": +1})
                                                                term3 = (0.5 / ez_sum) * az * swap_ERI({"az": -1, "cz": -1, "m": +1}) if az > 0 else 0.0
                                                                term4 = (0.5 / ez_sum) * bz * swap_ERI({"bz": -1, "cz": -1, "m": +1}) if bz > 0 else 0.0
                                                                term5 = (0.5 / eta) * (cz - 1) * (swap_ERI({"cz": -2}) - (zeta / ez_sum) * swap_ERI({"cz": -2, "m": +1})) if cz > 1 else 0.0
                                                                term6 = (0.5 / eta) * dz * (swap_ERI({"cz": -1, "dz": -1}) - (zeta / ez_sum) * swap_ERI({"cz": -1, "dz": -1, "m": +1})) if dz > 0 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif dx > 0:
                                                                term1 = QD[0] * swap_ERI({"dx": -1})
                                                                term2 = WQ[0] * swap_ERI({"dx": -1, "m": +1})
                                                                term3 = (0.5 / ez_sum) * ax * swap_ERI({"ax": -1, "dx": -1, "m": +1}) if ax > 0 else 0.0
                                                                term4 = (0.5 / ez_sum) * bx * swap_ERI({"bx": -1, "dx": -1, "m": +1}) if bx > 0 else 0.0
                                                                term5 = (0.5 / eta) * cx * (swap_ERI({"cx": -1, "dx": -1}) - (zeta / ez_sum) * swap_ERI({"cx": -1, "dx": -1, "m": +1})) if cx > 0 else 0.0
                                                                term6 = (0.5 / eta) * (dx - 1) * (swap_ERI({"dx": -2}) - (zeta / ez_sum) * swap_ERI({"dx": -2, "m": +1})) if dx > 1 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif dy > 0:
                                                                term1 = QD[1] * swap_ERI({"dy": -1})
                                                                term2 = WQ[1] * swap_ERI({"dy": -1, "m": +1})
                                                                term3 = (0.5 / ez_sum) * ay * swap_ERI({"ay": -1, "dy": -1, "m": +1}) if ay > 0 else 0.0
                                                                term4 = (0.5 / ez_sum) * by * swap_ERI({"by": -1, "dy": -1, "m": +1}) if by > 0 else 0.0
                                                                term5 = (0.5 / eta) * cy * (swap_ERI({"cy": -1, "dy": -1}) - (zeta / ez_sum) * swap_ERI({"cy": -1, "dy": -1, "m": +1})) if cy > 0 else 0.0
                                                                term6 = (0.5 / eta) * (dy - 1) * (swap_ERI({"dy": -2}) - (zeta / ez_sum) * swap_ERI({"dy": -2, "m": +1})) if dy > 1 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6

                                                            elif dz > 0:
                                                                term1 = QD[2] * swap_ERI({"dz": -1})
                                                                term2 = WQ[2] * swap_ERI({"dz": -1, "m": +1})
                                                                term3 = (0.5 / ez_sum) * az * swap_ERI({"az": -1, "dz": -1, "m": +1}) if az > 0 else 0.0
                                                                term4 = (0.5 / ez_sum) * bz * swap_ERI({"bz": -1, "dz": -1, "m": +1}) if bz > 0 else 0.0
                                                                term5 = (0.5 / eta) * cz * (swap_ERI({"cz": -1, "dz": -1}) - (zeta / ez_sum) * swap_ERI({"cz": -1, "dz": -1, "m": +1})) if cz > 0 else 0.0
                                                                term6 = (0.5 / eta) * (dz - 1) * (swap_ERI({"dz": -2}) - (zeta / ez_sum) * swap_ERI({"dz": -2, "m": +1})) if dz > 1 else 0.0
                                                                ERI[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, m] = term1 + term2 + term3 + term4 + term5 + term6
    return ERI

In [8]:
ERI = np.zeros((n_ao,n_ao,n_ao,n_ao))

# loop over the shells, basis.nshell is the number of shells
for a in range(basis.nshell()):
    for b in range(basis.nshell()):
        for c in range(basis.nshell()):
            for d in range(basis.nshell()):

                ashell = basis.shell(a) 
                bshell = basis.shell(b)
                cshell = basis.shell(c)
                dshell = basis.shell(d)

                AMa = ashell.am  # angular momenta
                AMb = bshell.am
                AMc = cshell.am
                AMd = dshell.am

                # Basis function index where the shell begins
                a_idx = ashell.function_index  
                b_idx = bshell.function_index
                c_idx = cshell.function_index
                d_idx = dshell.function_index

                a_coms = angular_combinations(AMa)
                b_coms = angular_combinations(AMb)
                c_coms = angular_combinations(AMc)
                d_coms = angular_combinations(AMd)

                counta = len(a_coms)
                countb = len(b_coms)
                countc = len(c_coms)
                countd = len(d_coms)

                if d < c:
                    ERI[a_idx:a_idx+counta, b_idx:b_idx+countb, c_idx:c_idx+countc, d_idx:d_idx+countd] = 1 * \
                    ERI[a_idx:a_idx+counta, b_idx:b_idx+countb, d_idx:d_idx+countd, c_idx:c_idx+countc].transpose((0, 1, 3, 2))
                    continue

                elif b < a:
                    ERI[a_idx:a_idx+counta, b_idx:b_idx+countb, c_idx:c_idx+countc, d_idx:d_idx+countd] = 1 * \
                    ERI[b_idx:b_idx+countb, a_idx:a_idx+counta, c_idx:c_idx+countc, d_idx:d_idx+countd].transpose((1, 0, 2, 3))
                    continue

                elif (a, b) > (c, d):
                    ERI[a_idx:a_idx+counta, b_idx:b_idx+countb, c_idx:c_idx+countc, d_idx:d_idx+countd] = 1 * \
                    ERI[c_idx:c_idx+countc, d_idx:d_idx+countd, a_idx:a_idx+counta, b_idx:b_idx+countb].transpose((2, 3, 0, 1))
                    continue

                # defining centers for each basis function 
                # mol.x() returns the x coordinate of the atom given by ishell.ncenter
                # we use this to define a coordinate vector for our centers
                A = np.array([mol.x(ashell.ncenter), mol.y(ashell.ncenter), mol.z(ashell.ncenter)])
                B = np.array([mol.x(bshell.ncenter), mol.y(bshell.ncenter), mol.z(bshell.ncenter)])
                C = np.array([mol.x(cshell.ncenter), mol.y(cshell.ncenter), mol.z(cshell.ncenter)])
                D = np.array([mol.x(dshell.ncenter), mol.y(dshell.ncenter), mol.z(dshell.ncenter)])

                # each shell has some number of primitives which make up each component of a shell
                # sto-3g has 3 primitives for every component of every shell.
                nprima = ashell.nprimitive 
                nprimb = bshell.nprimitive
                nprimc = cshell.nprimitive
                nprimd = dshell.nprimitive

                # loop over the primitives within a shell
                for a_prim in range(nprima):  
                    for b_prim in range(nprimb):
                        for c_prim in range(nprimc):
                            for d_prim in range(nprimd):
                                expa = ashell.exp(a_prim)
                                expb = bshell.exp(b_prim)
                                expc = cshell.exp(c_prim)
                                expd = dshell.exp(d_prim)

                                coefa = ashell.coef(a_prim)
                                coefb = bshell.coef(b_prim)
                                coefc = cshell.coef(c_prim)
                                coefd = dshell.coef(d_prim)

                                ERI_raw = get_ERIs(A, B, C, D, expa, expb, expc, expd, AMa, AMb, AMc, AMd)

                                zeta = expa + expb
                                xi = (expa * expb) / zeta
                                eta = expc + expd
                                xi_cd = (expc * expd) / eta # NOTE: In the original paper there is no notation for xi_cd

                                S_base_ab = (np.pi / zeta)**(3 / 2) * np.exp(-xi * np.sum((A-B)**2))
                                S_base_cd = (np.pi / eta)**(3 / 2) * np.exp(-xi_cd * np.sum((C-D)**2))

                                rho = (zeta * eta) / (zeta + eta)

                                ERI_base = 2 * (rho / np.pi)**0.5 * S_base_ab * S_base_cd

                                for counta, a_com in enumerate(a_coms):
                                    for countb, b_com in enumerate(b_coms):
                                        for countc, c_com in enumerate(c_coms):
                                            for countd, d_com in enumerate(d_coms):
                                                x, y, z = 0, 1, 2
                                                ax, ay, az = a_com
                                                bx, by, bz = b_com
                                                cx, cy, cz = c_com
                                                dx, dy, dz = d_com
                                                
                                                ERI_term = ERI_raw[ax, ay, az, bx, by, bz, cx, cy, cz, dx, dy, dz, 0]

                                                ERI[a_idx + counta, b_idx + countb, c_idx + countc, d_idx + countd] += ERI_base \
                                                                                                                    * coefa * coefb * coefc * coefd \
                                                                                                                    * ERI_term

KeyboardInterrupt: 

In [ ]:
Spsi4 = np.asarray(mints.ao_overlap())
Tpsi4 = np.asarray(mints.ao_kinetic())
Vpsi4 = np.asarray(mints.ao_potential())
ERIpsi4 = np.asarray(mints.ao_eri())

if np.allclose(S, Spsi4):
    print("S / Overlap integrals successful")
else:
    print("S / Overlap integrals failed")

if np.allclose(S, Spsi4):
    print("T / Kinetic integrals successful")
else:
    print("T / Kinetic integrals failed")

if np.allclose(V, Vpsi4):
    print("NAI / Nuclear attraction integrals successful")
else:
    print("NAI / Nuclear attraction integrals failed")

if np.allclose(ERI, ERIpsi4):
    print("ERI / Electron repulsion integrals successful")
else:
    print("ERI / Electron repulsion integrals failed")

S / Overlap integrals successful
T / Kinetic integrals successful
NAI / Nuclear attraction integrals successful
ERI / Electron repulsion integrals successful
